# Bài Tập Lớn Môn Học Máy — Dữ liệu Dạng Bảng## Travel Insurance Prediction Pipeline**Mục tiêu**: Xây dựng pipeline học máy toàn diện cho bài toán dự đoán mua bảo hiểm du lịch, bao gồm:- EDA (Khám phá dữ liệu)- Tiền xử lý (Imputation, Encoding, Scaling)- Trích xuất đặc trưng & Giảm chiều (PCA)- Huấn luyện & So sánh mô hình (Logistic Regression, SVM, Random Forest)- Tối ưu siêu tham số (Hyperparameter Tuning)- Deep Learning (MLP - PyTorch)- Đánh giá & Phân tích kết quả**Dataset**: [Travel Insurance Prediction Data (Kaggle)](https://www.kaggle.com/datasets/tejashvi14/travel-insurance-prediction-data)**GitHub**: [https://github.com/VietAnh1803/HCMUT_ML_Travel_Insurance_Pipeline](https://github.com/VietAnh1803/HCMUT_ML_Travel_Insurance_Pipeline)

## 1. Cài đặt thư viện & Tải dữ liệu

In [ ]:
# Cài đặt thư viện mở rộng
!pip install -q kagglehub torch matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import kagglehub
import os

# Tải bộ dữ liệu từ Kaggle
path = kagglehub.dataset_download("tejashvi14/travel-insurance-prediction-data")
print("Path to dataset files:", path)
csv_file = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')][0]

df = pd.read_csv(csv_file)
print("Số lượng mẫu:", df.shape[0])
print("Số lượng đặc trưng:", df.shape[1])
display(df.head())

## 2. Khám phá dữ liệu (EDA)

In [ ]:
# Loại bỏ cột Index/Unnamed nếu có
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# 2.1 Thông tin tổng quan
print("="*60)
print("THÔNG TIN TỔNG QUAN VỀ DỮ LIỆU")
print("="*60)
print(df.info())
print()
print(df.describe())
print()

# 2.2 Kiểm tra Missing Values
print("\nMissing Values:\n", df.isnull().sum())

# 2.3 Phân phối Target
plt.figure(figsize=(6, 4))
sns.countplot(x='TravelInsurance', data=df)
plt.title("Phân phối phân lớp Target (TravelInsurance)")
plt.show()

# 2.4 Phân tích đặc trưng số học (Numerical) & Phát hiện Outliers bằng Boxplot
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'TravelInsurance' in num_cols: num_cols.remove('TravelInsurance')

plt.figure(figsize=(12, 5))
for i, col in enumerate(num_cols):
    plt.subplot(1, len(num_cols), i+1)
    sns.boxplot(y=df[col])
    plt.title(f"Boxplot: {col}")
plt.tight_layout()
plt.show()

# 2.5 Ma trận tương quan (Correlation Heatmap)
plt.figure(figsize=(8, 6))
corr_matrix = df[num_cols + ['TravelInsurance']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 3. Chia tập dữ liệu (Train/Test Split)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['TravelInsurance'])
y = df['TravelInsurance']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## 4. Xây dựng Pipeline Tiền xử lý & Giảm chiều (PCA)**Các bước xử lý**:- **Imputation**: `KNNImputer` (numerical), `SimpleImputer` (categorical)- **Encoding**: `OneHotEncoder` cho biến phân loại- **Scaling**: So sánh `StandardScaler` vs `MinMaxScaler`- **Dimensionality Reduction**: PCA giữ lại 90% và 95% phương sai

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
from scipy.stats import randint, uniform

# Xác định cột số và cột phân loại
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numerical Features:", num_cols)
print("Categorical Features:", cat_cols)

# --- Tạo 4 bộ Preprocessor: 2 Scaler x 2 PCA threshold ---
def build_preprocessor(scaler, pca_threshold):
    num_pipeline = Pipeline(steps=[
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', scaler)
    ])
    cat_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])
    full_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=pca_threshold))
    ])
    return full_pipeline

# Xây dựng 4 cấu hình preprocessor
preprocessor_standard_95 = build_preprocessor(StandardScaler(), 0.95)
preprocessor_standard_90 = build_preprocessor(StandardScaler(), 0.90)
preprocessor_minmax_95 = build_preprocessor(MinMaxScaler(), 0.95)
preprocessor_minmax_90 = build_preprocessor(MinMaxScaler(), 0.90)

# Phân tích PCA Explained Variance
from sklearn.pipeline import Pipeline as Pipe

num_pipe_for_pca = Pipeline(steps=[('imputer', KNNImputer(n_neighbors=5)),('scaler', StandardScaler())])
cat_pipe_for_pca = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor_full = ColumnTransformer(transformers=[('num', num_pipe_for_pca, num_cols),('cat', cat_pipe_for_pca, cat_cols)])

X_train_processed = preprocessor_full.fit_transform(X_train)

pca_full = PCA().fit(X_train_processed)
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained_var)+1), explained_var, alpha=0.7, label='Từng thành phần')
plt.ylabel('Tỷ lệ phương sai giải thích')
plt.xlabel('Thành phần PCA')
plt.title('Explained Variance per Component')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, 'bo-')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
plt.axhline(y=0.90, color='g', linestyle='--', label='90% threshold')
plt.ylabel('Phương sai tích lũy')
plt.xlabel('Số thành phần PCA')
plt.title('Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Số thành phần giữ lại 90% phương sai: {np.argmax(cumulative_var >= 0.90) + 1}")
print(f"Số thành phần giữ lại 95% phương sai: {np.argmax(cumulative_var >= 0.95) + 1}")

## 5. Huấn luyện & So sánh các mô hìnhSo sánh **6 cấu hình** (3 mô hình × 2 scaler) với PCA 95%:

In [ ]:
# Định nghĩa 3 mô hình
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42, probability=True),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Định nghĩa 2 cấu hình scaler
scaler_configs = {
    "StandardScaler": preprocessor_standard_95,
    "MinMaxScaler": preprocessor_minmax_95,
}

# Bảng kết quả so sánh
results = []

for scaler_name, preproc in scaler_configs.items():
    for model_name, model in models.items():
        pipe = Pipeline(steps=[
            ('preprocessing', preproc),
            ('classifier', model)
        ])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        
        # Cross-Validation (5-fold)
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
        
        results.append({
            'Scaler': scaler_name,
            'Model': model_name,
            'Test Accuracy': round(acc, 4),
            'CV Mean Accuracy': round(cv_scores.mean(), 4),
            'CV Std': round(cv_scores.std(), 4),
            'Precision (macro)': round(report['macro avg']['precision'], 4),
            'Recall (macro)': round(report['macro avg']['recall'], 4),
            'F1 (macro)': round(report['macro avg']['f1-score'], 4),
        })
        print(f"[{scaler_name}] {model_name}: Test Acc={acc:.4f}, CV Acc={cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Hiển thị bảng kết quả tổng hợp
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("BẢNG TỔNG HỢP KẾT QUẢ SO SÁNH CÁC MÔ HÌNH")
print("="*80)
display(results_df)

# Xác định mô hình tốt nhất
best_row = results_df.loc[results_df['Test Accuracy'].idxmax()]
print(f"\nMô hình tốt nhất: {best_row['Model']} với {best_row['Scaler']}")
print(f"  Test Accuracy: {best_row['Test Accuracy']}")
print(f"  CV Mean Accuracy: {best_row['CV Mean Accuracy']} ± {best_row['CV Std']}")

## 6. Tối ưu siêu tham số (Hyperparameter Tuning)Sử dụng `RandomizedSearchCV` cho mô hình Random Forest (mô hình tốt nhất):

In [ ]:
# RandomizedSearchCV cho Random Forest
param_dist = {
    'classifier__n_estimators': randint(50, 300),
    'classifier__max_depth': [None, 5, 10, 20, 30],
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf': randint(1, 10),
    'classifier__max_features': ['sqrt', 'log2', None]
}

rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor_standard_95),
    ('classifier', RandomForestClassifier(random_state=42))
])

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"\nBest Parameters: {random_search.best_params_}")
print(f"Best CV Accuracy: {random_search.best_score_:.4f}")

# Đánh giá mô hình đã tối ưu trên tập Test
y_pred_tuned = random_search.best_estimator_.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned RF Test Accuracy: {tuned_acc:.4f}")
print("\nClassification Report (Tuned RF):")
print(classification_report(y_test, y_pred_tuned))

## 7. Đánh giá chi tiết### Confusion Matrix & ROC Curve

In [ ]:
# --- Confusion Matrix cho mô hình tốt nhất (Tuned RF) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Không mua (0)', 'Mua (1)'],
            yticklabels=['Không mua (0)', 'Mua (1)'])
axes[0].set_title('Confusion Matrix - Tuned Random Forest')
axes[0].set_ylabel('Thực tế')
axes[0].set_xlabel('Dự đoán')

# ROC Curve
y_prob_tuned = random_search.best_estimator_.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_tuned)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Tuned Random Forest')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

## 8. Phân tích kết quả & Nhận xét**Ưu điểm**:- Pipeline tự động hóa toàn bộ quy trình từ tiền xử lý đến đánh giá- `KNNImputer` xử lý missing values thông minh hơn so với đơn giản thay thế bằng mean/median- PCA giảm chiều hiệu quả, giữ lại 95% phương sai giải thích- `RandomizedSearchCV` giúp tối ưu siêu tham số một cách tự động**Hạn chế**:- Dataset nhỏ (< 2000 mẫu) nên kết quả Cross-Validation có phương sai khá cao- Một số đặc trưng phân loại (categorical) có ít giá trị unique, có thể thử LabelEncoder thay vì OneHotEncoder- Chưa thử nghiệm các kỹ thuật Oversampling (SMOTE) cho dữ liệu mất cân bằng**So sánh Scaler**:- `StandardScaler` thường cho kết quả tốt hơn `MinMaxScaler` trong trường hợp này, do SVM và Logistic Regression hội tụ tốt hơn với dữ liệu chuẩn hóa theo phân phối chuẩn

## 9. (Điểm cộng) Deep Learning Pipeline — Mạng MLP (PyTorch)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --- Chuẩn bị dữ liệu cho Deep Learning ---
# Transform dữ liệu bằng preprocessor đã fit
X_train_dl = preprocessor_standard_95.fit_transform(X_train)
X_test_dl = preprocessor_standard_95.transform(X_test)

class TravelDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TravelDataset(X_train_dl, y_train)
test_dataset = TravelDataset(X_test_dl, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- Kiến trúc MLP ---
input_dim = X_train_dl.shape[1]

class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.network(x)

model_dl = MLPClassifier(input_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_dl.parameters(), lr=0.001)

# --- Huấn luyện ---
num_epochs = 30
for epoch in range(num_epochs):
    model_dl.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_dl(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        model_dl.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model_dl(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        val_acc = correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f} - Val Accuracy: {val_acc:.4f}")

# --- Đánh giá Deep Learning ---
model_dl.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model_dl(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

print("\nBáo cáo phân loại Mạng Deep Learning MLP:")
print(classification_report(all_labels, all_preds))

## 10. Lưu đặc trưng & Mô hình

In [ ]:
import joblib

# Xử lý đường dẫn tương thích giữa Local và Colab
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    features_dir = os.path.join(os.path.dirname(current_dir), 'features')
else:
    features_dir = os.path.join(current_dir, 'features')

os.makedirs(features_dir, exist_ok=True)

# --- 10.1 Lưu vector đặc trưng (yêu cầu đề bài: .npy) ---
np.save(os.path.join(features_dir, 'X_train_features.npy'), X_train_dl)
np.save(os.path.join(features_dir, 'X_test_features.npy'), X_test_dl)
np.save(os.path.join(features_dir, 'y_train.npy'), y_train.values)
np.save(os.path.join(features_dir, 'y_test.npy'), y_test.values)
print(f"Đã lưu đặc trưng .npy vào: {features_dir}")

# --- 10.2 Lưu Scikit-Learn Pipeline (Tuned RF) ---
save_path = os.path.join(features_dir, 'travel_insurance_pipeline.pkl')
joblib.dump(random_search.best_estimator_, save_path)
print(f"Đã lưu Scikit-Learn Pipeline (Tuned RF) vào: {save_path}")

# --- 10.3 Lưu PyTorch MLP Model ---
mlp_save_path = os.path.join(features_dir, 'travel_insurance_mlp.pth')
torch.save(model_dl.state_dict(), mlp_save_path)
print(f"Đã lưu PyTorch MLP Model vào: {mlp_save_path}")